In [4]:
!pip install kagglehub==0.3.13 vllm==0.10.0 logits-processor-zoo==0.1.10 triton==3.2.0 clean-text bitsandbytes peft accelerate datasets emoji setuptools>=40.8.0 numpy<2

Using Python 3.12.11 environment at: /usr
  × No solution found when resolving dependencies:
  ╰─▶ Because torch==2.7.1 depends on triton{platform_machine == 'x86_64'
      and sys_platform == 'linux'}==3.3.1 and vllm==0.10.0 depends
      on torch==2.7.1, we can conclude that vllm==0.10.0 depends on
      triton==3.3.1.
      And because you require vllm==0.10.0 and triton==3.2.0, we can conclude
      that your requirements are unsatisfiable.


In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

jigsaw_agile_community_rules_path = kagglehub.competition_download('jigsaw-agile-community-rules')
bibanh_qwen2_5_32b_gptq_int4_batch4_full_path = kagglehub.dataset_download('bibanh/qwen2-5-32b-gptq-int4-batch4-full')
bibanh_qwen3_8b_embedding_path = kagglehub.dataset_download('bibanh/qwen3-8b-embedding')
mks2192_jigsaw_llama3_1_8b_instruct_training_one_epoch_path = kagglehub.notebook_output_download('mks2192/jigsaw-llama3-1-8b-instruct-training-one-epoch')
fuumin621_qwen2_5_32b_instruct_gptq_int4_path = kagglehub.notebook_output_download('fuumin621/qwen2-5-32b-instruct-gptq-int4')
# hiranorm_jigsaw_packages_path = kagglehub.notebook_output_download('hiranorm/jigsaw-packages')
qwen_lm_qwen_3_embedding_transformers_0_6b_1_path = kagglehub.model_download('qwen-lm/qwen-3-embedding/Transformers/0.6b/1')

print('Data source import complete.')



### References

*   [https://www.kaggle.com/code/abdmental01/jigsaw-mpnet-base-v2-inference-cv-0-876](https://www.kaggle.com/code/abdmental01/jigsaw-mpnet-base-v2-inference-cv-0-876)
*   [https://www.kaggle.com/code/aerdem4/jigsaw-acrc-qwen7b-finetune-logits-processor-zoo](https://www.kaggle.com/code/aerdem4/jigsaw-acrc-qwen7b-finetune-logits-processor-zoo)
*   [https://www.guruguru.science/competitions/24/discussions/21027ff1-2074-4e21-a249-b2d4170bd516/](https://www.guruguru.science/competitions/24/discussions/21027ff1-2074-4e21-a249-b2d4170bd516/)
*   https://www.kaggle.com/code/mks2192/jigsaw-llama3-1-8b-instruct-training-one-epoch
*   [https://www.kaggle.com/code/fuumin621/qwen2-5-lora-finetune-baseline-inference](https://www.kaggle.com/code/fuumin621/qwen2-5-lora-finetune-baseline-inference)
*   https://www.kaggle.com/code/neibyr/30-min-just-use-semantic-search-qwen3-emb-0-6b

### I want to say thanks to @neibyr for your interesting idea: [Retrieve by Qwen3Embedding](http://https://www.kaggle.com/code/neibyr/30-min-just-use-semantic-search-qwen3-emb-0-6b)

# 1. Qwen2.5 32B GPTQ Int4 Inference

In [4]:
! mkdir -p /tmp/src

In [5]:
%%writefile /tmp/src/infer_qwen.py

import os
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
import torch
import vllm
import numpy as np
from vllm.lora.request import LoRARequest
import argparse
from scipy.special import softmax
import re
import kagglehub # Import kagglehub

# Use kagglehub.get_path to get the path to the competition data
# jigsaw_agile_community_rules_path = kagglehub.get_path('jigsaw-agile-community-rules')
df = pd.read_csv("/root/.cache/kagglehub/competitions/jigsaw-agile-community-rules/test.csv")

MODEL_NAME = '/root/.cache/kagglehub/notebooks/fuumin621/qwen2-5-32b-instruct-gptq-int4/output/versions/3'
LORA_PATH = '/kaggle/input/qwen2-5-32b-gptq-int4-batch4-full'
if __name__=='__main__':
    os.environ["VLLM_USE_V1"] = "0"

    llm = vllm.LLM(
        MODEL_NAME,
        # quantization='awq',
        quantization='gptq',
        tensor_parallel_size=torch.cuda.device_count(),
        gpu_memory_utilization=0.95,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=4096,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
    )
    tokenizer = llm.get_tokenizer()
    SYS_PROMPT = """
    You are given a comment on reddit. Your task is to classify if it violates the given rule.Think step by step about your reasoning, then provide your final answer
  as Yes or No.
    """

    prompts = []
    for i, row in df.iterrows():
        text = f"""
    r/{row.subreddit}
    Rule: {row.rule}

    1) {row.positive_example_1}
    Violation: Yes

    2) {row.positive_example_2}
    Violation: Yes

    3) {row.negative_example_1}
    Violation: No

    4) {row.negative_example_2}
    Violation: No

    5) {row.body}
    """

        messages = [
            {"role": "system", "content": SYS_PROMPT},
            {"role": "user", "content": text}
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        ) + "Let me think step by step:\n\nAnswer:"

        prompts.append(prompt)

    df["prompt"] = prompts

    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=['Yes','No'])
    outputs = llm.generate(
      prompts,
      vllm.SamplingParams(
          skip_special_tokens=True,
          max_tokens=100,
          temperature=0.1,  # Add some temperature for better reasoning
          # Remove logits_processors=[mclp] - this is causing the error
          logprobs=2,
      ),
      use_tqdm=True,
      lora_request=LoRARequest("default", 1, LORA_PATH)
  )

  # Parse answers from generated text instead of using logits processor
    def extract_final_answer(text):
        # Look for "Answer: Yes" or "Answer: No" pattern
        match = re.search(r'Answer:\s*(Yes|No)', text, re.IGNORECASE)
        if match:
            return match.group(1).lower()
        # Fallback: look for last Yes/No in the text
        matches = re.findall(r'\b(Yes|No)\b', text, re.IGNORECASE)
        if matches:
            return matches[-1].lower()
        return "no"  # Default fallback

  # Extract predictions from reasoning text
    predictions = []
    for out in outputs:
        generated_text = out.outputs[0].text
        print(f"Generated: {generated_text}")  # Debug output
        final_answer = extract_final_answer(generated_text)
        predictions.append(1.0 if final_answer == "yes" else 0.0)

    df["pred"] = predictions
    df['rule_violation'] = df["pred"]
    df[['row_id', 'rule_violation']].to_csv("submission_qwen.csv", index=False)

Overwriting /tmp/src/infer_qwen.py


In [6]:
%cd /tmp
!python src/infer_qwen.py

/tmp
2025-09-04 10:44:19.361735: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-04 10:44:19.382038: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756982659.405593   74806 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756982659.412583   74806 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756982659.430784   74806 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid lin

In [7]:
jigsaw_agile_community_rules_path

'/root/.cache/kagglehub/competitions/jigsaw-agile-community-rules'

# 2. Qwen3 0.6b Embedding

In [8]:
qwen_lm_qwen_3_embedding_transformers_0_6b_1_path

'/kaggle/input/qwen-3-embedding/transformers/0.6b/1'

In [9]:
import os
import pandas as pd

In [10]:
%%writefile constants.py
EMBDEDDING_MODEL_PATH = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"
MODEL_OUTPUT_PATH = '/kaggle/input/qwen3-8b-embedding'
DATA_PATH = "/root/.cache/kagglehub/competitions/jigsaw-agile-community-rules"

# https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/blob/main/config_sentence_transformers.json
EMBEDDING_MODEL_QUERY = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"

CLEAN_TEXT = True
TOP_K = 2000
BATCH_SIZE = 128


Overwriting constants.py


In [11]:
%%writefile utils.py
import pandas as pd
import torch.distributed as dist

from datasets import Dataset
from cleantext import clean
from tqdm.auto import tqdm

from constants import CLEAN_TEXT


def build_prompt(row):
    return f"""r/{row["subreddit"]}\nComment: {row["body"]}"""


def cleaner(text):
    return clean(
        text,
        fix_unicode=True,
        to_ascii=True,
        lower=False,
        no_line_breaks=False,
        no_urls=True,
        no_emails=True,
        no_phone_numbers=True,
        no_numbers=False,
        no_digits=False,
        no_currency_symbols=False,
        no_punct=False,
        replace_with_url="<URL>",
        replace_with_email="<EMAIL>",
        replace_with_phone_number="<PHONE>",
        lang="en",
    )



def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = []
    flatten.append(train_dataset[["body", "rule", "subreddit", "rule_violation"]])

    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)

    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(ignore_index=True)
    return dataframe


def prepare_dataframe(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)


    if CLEAN_TEXT:
        tqdm.pandas(desc="cleaner")
        dataframe["prompt"] = dataframe["prompt"].progress_apply(cleaner)

    if "rule_violation" in dataframe.columns:
        dataframe["rule_violation"] = dataframe["rule_violation"].map(
            {
                1: 1,
                0: -1,
            }
        )

    return dataframe

Overwriting utils.py


In [12]:
%%writefile semantic.py
import pandas as pd
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import semantic_search, dot_score
from tqdm.auto import tqdm
from peft import PeftModel, PeftConfig


from utils import get_dataframe_to_train, prepare_dataframe
from constants import DATA_PATH, EMBDEDDING_MODEL_PATH, EMBEDDING_MODEL_QUERY, TOP_K, BATCH_SIZE, MODEL_OUTPUT_PATH



def get_scores(test_dataframe):
    corpus_dataframe = get_dataframe_to_train(DATA_PATH)
    corpus_dataframe = prepare_dataframe(corpus_dataframe)

    # Load base model
    model = AutoModelForCausalLM.from_pretrained(EMBDEDDING_MODEL_PATH)
    tokenizer = AutoTokenizer.from_pretrained(EMBDEDDING_MODEL_PATH)

    # Load adapter configuration and model
    adapter_config = PeftConfig.from_pretrained(MODEL_OUTPUT_PATH)
    lora_model = PeftModel.from_pretrained(model, MODEL_OUTPUT_PATH, config=adapter_config)
    merged_model = lora_model.merge_and_unload()
    tokenizer.save_pretrained("Qwen3Emb_Finetuned")
    merged_model.save_pretrained("Qwen3Emb_Finetuned")

    # 4. Tạo lại SentenceTransformer từ encoder đã merge
    embedding_model = SentenceTransformer(model_name_or_path="Qwen3Emb_Finetuned", device="cuda")

    print('Done loading model!')

    result = []
    for rule in tqdm(test_dataframe["rule"].unique(), desc=f"Generate scores for each rule"):
        test_dataframe_part = test_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_dataframe_part = corpus_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_dataframe_part = corpus_dataframe_part.reset_index(names="row_id")

        query_embeddings = embedding_model.encode(
            sentences=test_dataframe_part["prompt"].tolist(),
            prompt=EMBEDDING_MODEL_QUERY,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        document_embeddings = embedding_model.encode(
            sentences=corpus_dataframe_part["prompt"].tolist(),
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        test_dataframe_part["semantic"] = semantic_search(
            query_embeddings,
            document_embeddings,
            top_k=TOP_K,
            score_function=dot_score,
        )
        def get_score(semantic):
            semantic = pd.DataFrame(semantic)
            semantic = semantic.merge(
                corpus_dataframe_part[["row_id", "rule_violation"]],
                how="left",
                left_on="corpus_id",
                right_on="row_id",
            )
            semantic["score"] = semantic["score"]*semantic["rule_violation"]
            return semantic["score"].sum()

        tqdm.pandas(desc=f"Add label for {rule=}")
        test_dataframe_part["rule_violation"] = test_dataframe_part["semantic"].progress_apply(get_score)
        result.append(test_dataframe_part[["row_id", "rule_violation"]].copy())

    submission = pd.concat(result, axis=0)
    return submission


def generate_submission():
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    test_dataframe = prepare_dataframe(test_dataframe)

    submission = get_scores(test_dataframe)
    submission = test_dataframe[["row_id"]].merge(submission, on="row_id", how="left")
    submission.to_csv("submission_qwen3.csv", index=False)


if __name__ == "__main__":
    generate_submission()



Overwriting semantic.py


In [13]:
!python /tmp/semantic.py

2025-09-04 10:45:23.697193: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-04 10:45:23.716086: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756982723.737499   75232 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756982723.744204   75232 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756982723.761446   75232 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

# 3. Llama 3.1 8B

In [14]:
import os, math, numpy as np
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [15]:
import pandas as pd
import numpy as np

test = pd.read_csv('/root/.cache/kagglehub/competitions/jigsaw-agile-community-rules/test.csv')
sub = pd.read_csv('/root/.cache/kagglehub/competitions/jigsaw-agile-community-rules/sample_submission.csv', index_col='row_id')
sub


,rule_violation
row_id,
2029,0.5
2030,0.5
2031,0.5
2032,0.5
2033,0.5
2034,0.5
2035,0.5
2036,0.5
2037,0.5


In [16]:
!ls $mks2192_jigsaw_llama3_1_8b_instruct_training_one_epoch_path

__huggingface_repos__.json  llama-8b-instruct-jigsaw


In [38]:


import vllm

llm = vllm.LLM(
    "/root/.cache/kagglehub/notebooks/mks2192/jigsaw-llama3-1-8b-instruct-training-one-epoch/output/versions/2/llama-8b-instruct-jigsaw",
    tensor_parallel_size=1,
    gpu_memory_utilization=0.95,
    trust_remote_code=True,
    dtype="half",
    enforce_eager=True,
    max_model_len=2048,
    # disable_log_stats=True,
    # enable_prefix_caching=True,

)
tokenizer = llm.get_tokenizer()



INFO 09-04 10:56:32 [config.py:1604] Using max model len 2048
WARNING 09-04 10:56:32 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 09-04 10:56:32 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='/root/.cache/kagglehub/notebooks/mks2192/jigsaw-llama3-1-8b-instruct-training-one-epoch/output/versions/2/llama-8b-instruct-jigsaw', speculative_config=None, tokenizer='/root/.cache/kagglehub/notebooks/mks2192/jigsaw-llama3-1-8b-instruct-training-one-epoch/output/versions/2/llama-8b-instruct-jigsaw', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=aut

OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 10.25 MiB is free. Process 900967 has 1.38 GiB memory in use. Process 908511 has 38.15 GiB memory in use. Of the allocated memory 1002.00 MiB is allocated by PyTorch, and 0 bytes is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [18]:
from typing import Any, Dict, List
from transformers import LogitsProcessor
import torch

choices = ["No", "Yes"]

KEEP = []
for x in choices:
    c = tokenizer.encode(x,add_special_tokens=False)[0]
    KEEP.append(c)
print(f"Force predictions to be tokens {KEEP} which are {choices}.")

class DigitLogitsProcessor(LogitsProcessor):
    def __init__(self, tokenizer):
        self.allowed_ids = KEEP

    def __call__(self, input_ids: List[int], scores: torch.Tensor) -> torch.Tensor:
        scores[self.allowed_ids] += 100
        return scores

Force predictions to be tokens [2822, 9642] which are ['No', 'Yes'].


In [19]:
sys_prompt = '''You are given a comment on reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''

In [20]:
def formatting(dataset):
    texts = []
    for i in range(len(dataset)):
        texts.append(tokenizer.apply_chat_template(dataset[i], tokenize=False, add_generation_prompt=False))
    return texts


In [21]:
template = """
Subreddit: r/{subreddit}
Rule: {rule}
Examples:
1) {positive_example_1}
Violation: Yes

2) {negative_example_1}
Violation: No

3) {negative_example_2}
Violation: No

4) {positive_example_2}
Violation: Yes
Comment:
{body}
Violation: """

In [22]:
dataset = []
for index,row in test.iterrows():

    formatted_sample = [
        {
        "role": "system",
        "content": sys_prompt
    },
       {
           "role": "user",
           "content": template.format(
               rule = row.rule,
               subreddit = row.subreddit,
               body = row.body,
               positive_example_1 = row.positive_example_1,
               negative_example_1 = row.negative_example_1,
               positive_example_2 = row.positive_example_2,
               negative_example_2 = row.negative_example_2
           )
       }]

    dataset.append( formatted_sample )



In [37]:
all_prompts = formatting(dataset)
import os
os.environ["VLLM_USE_V0"] = "1"

In [36]:
logits_processors = [DigitLogitsProcessor(tokenizer)]
responses = llm.generate(
    all_prompts,
    vllm.SamplingParams(
        n=1,  # Number of output sequences to return for each prompt.
        top_p=0.9,  # Float that controls the cumulative probability of the top tokens to consider.
        temperature=0,  # randomness of the sampling
        seed=777, # Seed for reprodicibility
        skip_special_tokens=True,  # Whether to skip special tokens in the output.
        max_tokens=1,  # Maximum number of tokens to generate per output sequence.
        logits_processors=logits_processors,
        logprobs = 2
    ),
    use_tqdm = True
)

Adding requests:   0%|          | 0/10 [00:00<?, ?it/s]

ValueError: vLLM V1 does not support per request user provided logits processors.

In [26]:
results = []
errors = 0

for i,response in enumerate(responses):
    try:
        x = response.outputs[0].logprobs[0]
        logprobs = []
        for k in KEEP:
            if k in x:
                logprobs.append( math.exp(x[k].logprob) )
            else:
                logprobs.append( 0 )
                print(f"bad logits {i}")
        logprobs = np.array( logprobs )
        logprobs /= logprobs.sum()
        results.append( logprobs )
    except:
        #print(f"error {i}")
        results.append( np.array([1/2., 1/2.]) )
        errors += 1

print(f"There were {errors} inference errors out of {i+1} inferences")
results = np.vstack(results)

bad logits 0
bad logits 0
bad logits 1
bad logits 1
bad logits 2
bad logits 2
bad logits 3
bad logits 3
bad logits 4
bad logits 4
bad logits 5
bad logits 5
bad logits 6
bad logits 6
bad logits 7
bad logits 7
bad logits 8
bad logits 8
bad logits 9
bad logits 9
There were 10 inference errors out of 10 inferences


In [27]:


probs = [x[1] for x in results]
sub['rule_violation'] = probs
sub.to_csv('submission_llama.csv')



# 4. ENSEMBLE RESULT

In [28]:
import pandas as pd
import numpy as np

q = pd.read_csv('submission_qwen.csv')
l = pd.read_csv('submission_qwen3.csv')
m = pd.read_csv('submission_llama.csv')

rq = q['rule_violation'].rank(method='average') / (len(q)+1)
rl = l['rule_violation'].rank(method='average') / (len(l)+1)
rm = m['rule_violation'].rank(method='average') / (len(m)+1)


blend = 0.5*rq + 0.4*rl + 0.1*rm   # or tune the rank-weights with a tiny grid using OOF
q['rule_violation'] = blend
q.to_csv('submission.csv', index=False)
